# Data aggregation for 2026 matches

In [53]:
import joblib
import pandas as pd

Load data

In [54]:
df_team_stats = pd.read_csv("model/team_stats.csv")
df_matches = pd.read_csv("matches/2026_group_matches.csv")

Join team features to matches

In [55]:
matches = df_matches.merge(
    df_team_stats,
    left_on="team1",
    right_on="team",
    how="left"
)

matches = matches.rename(
    columns={col: f"{col}_team1" for col in df_team_stats.columns if col != "team"}
)
matches = matches.drop(columns=["team"], errors="ignore")


matches = matches.merge(
    df_team_stats,
    left_on="team2",
    right_on="team",
    how="left"
)

matches = matches.rename(
    columns={col: f"{col}_team2" for col in df_team_stats.columns if col != "team"}
)
matches = matches.drop(columns=["team"], errors="ignore")


matches.head()

,match_id,team1,team2,date,group,total_goals_team1,num_players_with_goals_team1,max_goals_by_single_player_team1,total_matches_team1,matches_with_goals_team1,matches_without_goals_team1,avg_goals_per_match_team1,total_goals_team2,num_players_with_goals_team2,max_goals_by_single_player_team2,total_matches_team2,matches_with_goals_team2,matches_without_goals_team2,avg_goals_per_match_team2
0,M-2026-01,Mexico,South Africa,12.6.2026,A,23.0,15.0,4.0,23.0,15.0,8.0,1.000000,8.0,8.0,1.0,6.0,5.0,1.0,1.333333
1,M-2026-02,South Korea,Czech Republic,12.6.2026,A,28.0,18.0,3.0,24.0,18.0,6.0,1.166667,3.0,2.0,2.0,3.0,1.0,2.0,1.000000
2,M-2026-03,Canada,Bosnia and Herzegovina,13.6.2026,B,2.0,2.0,1.0,3.0,2.0,1.0,0.666667,4.0,4.0,1.0,3.0,2.0,1.0,1.333333
3,M-2026-04,United States,Paraguay,13.6.2026,D,22.0,14.0,5.0,20.0,16.0,4.0,1.100000,11.0,9.0,3.0,12.0,6.0,6.0,0.916667
4,M-2026-05,Qatar,Switzerland,14.6.2026,B,1.0,1.0,1.0,3.0,1.0,2.0,0.333333,22.0,14.0,5.0,19.0,12.0,7.0,1.157895


Handle new comers, who qualified for the first time 

In [56]:
global_means = df_team_stats.mean(numeric_only=True)

for col in global_means.index:
    matches[f"{col}_team1"] = matches[f"{col}_team1"].fillna(global_means[col])
    matches[f"{col}_team2"] = matches[f"{col}_team2"].fillna(global_means[col])

assert matches.isna().sum().sum() == 0
matches.head()

,match_id,team1,team2,date,group,total_goals_team1,num_players_with_goals_team1,max_goals_by_single_player_team1,total_matches_team1,matches_with_goals_team1,matches_without_goals_team1,avg_goals_per_match_team1,total_goals_team2,num_players_with_goals_team2,max_goals_by_single_player_team2,total_matches_team2,matches_with_goals_team2,matches_without_goals_team2,avg_goals_per_match_team2
0,M-2026-01,Mexico,South Africa,12.6.2026,A,23.0,15.0,4.0,23.0,15.0,8.0,1.000000,8.0,8.0,1.0,6.0,5.0,1.0,1.333333
1,M-2026-02,South Korea,Czech Republic,12.6.2026,A,28.0,18.0,3.0,24.0,18.0,6.0,1.166667,3.0,2.0,2.0,3.0,1.0,2.0,1.000000
2,M-2026-03,Canada,Bosnia and Herzegovina,13.6.2026,B,2.0,2.0,1.0,3.0,2.0,1.0,0.666667,4.0,4.0,1.0,3.0,2.0,1.0,1.333333
3,M-2026-04,United States,Paraguay,13.6.2026,D,22.0,14.0,5.0,20.0,16.0,4.0,1.100000,11.0,9.0,3.0,12.0,6.0,6.0,0.916667
4,M-2026-05,Qatar,Switzerland,14.6.2026,B,1.0,1.0,1.0,3.0,1.0,2.0,0.333333,22.0,14.0,5.0,19.0,12.0,7.0,1.157895


Compute diff features

In [57]:
matches["goals_diff"] = matches["total_goals_team1"] - matches["total_goals_team2"]
matches["matches_diff"] = matches["total_matches_team1"] - matches["total_matches_team2"]
matches["avg_goals_diff"] = matches["avg_goals_per_match_team1"] - matches["avg_goals_per_match_team2"]
matches["num_players_with_goals_diff"] = matches["num_players_with_goals_team1"] - matches["num_players_with_goals_team2"]
matches["max_goals_diff"] = matches["max_goals_by_single_player_team1"] - matches["max_goals_by_single_player_team2"]

matches = matches.drop(columns=["team_team1", "team_team2"], errors="ignore")

matches.head()

,match_id,team1,team2,date,group,total_goals_team1,num_players_with_goals_team1,max_goals_by_single_player_team1,total_matches_team1,matches_with_goals_team1,...,max_goals_by_single_player_team2,total_matches_team2,matches_with_goals_team2,matches_without_goals_team2,avg_goals_per_match_team2,goals_diff,matches_diff,avg_goals_diff,num_players_with_goals_diff,max_goals_diff
0,M-2026-01,Mexico,South Africa,12.6.2026,A,23.0,15.0,4.0,23.0,15.0,...,1.0,6.0,5.0,1.0,1.333333,15.0,17.0,-0.333333,7.0,3.0
1,M-2026-02,South Korea,Czech Republic,12.6.2026,A,28.0,18.0,3.0,24.0,18.0,...,2.0,3.0,1.0,2.0,1.000000,25.0,21.0,0.166667,16.0,1.0
2,M-2026-03,Canada,Bosnia and Herzegovina,13.6.2026,B,2.0,2.0,1.0,3.0,2.0,...,1.0,3.0,2.0,1.0,1.333333,-2.0,0.0,-0.666667,-2.0,0.0
3,M-2026-04,United States,Paraguay,13.6.2026,D,22.0,14.0,5.0,20.0,16.0,...,3.0,12.0,6.0,6.0,0.916667,11.0,8.0,0.183333,5.0,2.0
4,M-2026-05,Qatar,Switzerland,14.6.2026,B,1.0,1.0,1.0,3.0,1.0,...,5.0,19.0,12.0,7.0,1.157895,-21.0,-16.0,-0.824561,-13.0,-4.0


# Prediction for 2026 matches

Load model

In [58]:
clf = joblib.load("model/worldcup_model.pkl")
feature_cols = joblib.load("model/feature_columns.pkl")

Predict winning probability for 2026 matches

In [59]:
X = matches[feature_cols]
matches["pred_prob_win"] = clf.predict_proba(X)[:, 1]

predictions = matches.loc[:, [
    "match_id",
    "team1",
    "team2",
    "date",
    "group",
    "pred_prob_win"
]].copy()

predictions["predicted_winner"] = predictions.apply(
    lambda row: row["team1"] if row["pred_prob_win"] >= 0.5 else row["team2"],
    axis=1
)

predictions["confidence"] = predictions["pred_prob_win"].apply(
    lambda p: p if p >= 0.5 else 1 - p
)

predictions = predictions.sort_values(
    by=["group", "confidence"],
    ascending=[True, False]
)

predictions.head(6*4)

,match_id,team1,team2,date,group,pred_prob_win,predicted_winner,confidence
24,M-2026-25,Czech Republic,South Africa,18.6.2026,A,0.120000,South Africa,0.880000
0,M-2026-01,Mexico,South Africa,12.6.2026,A,0.140000,South Africa,0.860000
27,M-2026-28,Mexico,South Korea,19.6.2026,A,0.310417,South Korea,0.689583
53,M-2026-54,Czech Republic,Mexico,25.6.2026,A,0.542500,Czech Republic,0.542500
52,M-2026-53,South Africa,South Korea,25.6.2026,A,0.468000,South Korea,0.532000
1,M-2026-02,South Korea,Czech Republic,12.6.2026,A,0.507500,South Korea,0.507500
4,M-2026-05,Qatar,Switzerland,14.6.2026,B,0.040000,Switzerland,0.960000
2,M-2026-03,Canada,Bosnia and Herzegovina,13.6.2026,B,0.150000,Bosnia and Herzegovina,0.850000
49,M-2026-50,Bosnia and Herzegovina,Qatar,25.6.2026,B,0.691667,Bosnia and Herzegovina,0.691667
25,M-2026-26,Switzerland,Bosnia and Herzegovina,19.6.2026,B,0.314167,Bosnia and Herzegovina,0.685833


Output predictions

In [60]:
predictions.to_csv(
    "predictions/predictions_2026_group_matches.csv",
    index=False
)

Output confidence

In [61]:
predictions["confidence_bucket"] = pd.cut(
    predictions["confidence"],
    bins=[0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
)

predictions["confidence_bucket"].value_counts().sort_index()

confidence_bucket
(0.5, 0.6]    16
(0.6, 0.7]    14
(0.7, 0.8]    12
(0.8, 0.9]    16
(0.9, 1.0]    14
Name: count, dtype: int64